# GRASP Library Designer

Redesign and anneal the **42-module combinatorial GRASP library** (Farley et al., *NAR* 2025), then compile a target RNA with GAP.

1. Run **0 · Install** once  
2. Fill the forms and run cells top → bottom  

Code is hidden by default (Colab Forms).


In [ ]:
#@title 0 · Install (PyPI) { display-mode: "form" }
#@markdown Installs everything from PyPI. No GitHub token needed. Re-run if imports fail after a runtime restart.

%pip install -q -U --force-reinstall "grasp-library-designer>=0.1.14"
import importlib, sys
for _m in [m for m in list(sys.modules) if m == 'grasp_library' or m.startswith('grasp_library.')]:
    del sys.modules[_m]


import importlib
import grasp_library
from importlib.metadata import version

print("grasp-library-designer", version("grasp-library-designer"))
print("import ok:", grasp_library.__name__)
from grasp_library import notebook_ui as _ui
print("kazusa_codon_reminder:", hasattr(_ui, "kazusa_codon_reminder"))


In [ ]:
#@title 1 · Settings { display-mode: "form" }
#@markdown Organism, synthesis, ligation, architecture, overhangs, and optimization depth — then run this cell.
#@markdown **Codon tables:** browse [Kazusa CUTG](https://www.kazusa.or.jp/codon/) (search → copy the `species=` accession from the URL). Built-ins cover common hosts; otherwise choose *Fetch from Kazusa* or *Upload your own*.

target_rna = "UUACACGUG" #@param {type:"string"}
organism = "Escherichia coli (Kazusa)" #@param ["Escherichia coli (Kazusa)", "Saccharomyces cerevisiae (Kazusa)", "Homo sapiens (Kazusa)", "Euglena gracilis nuclear (Kazusa)", "Chlamydomonas reinhardtii nuclear (Kazusa)", "Chlamydomonas reinhardtii chloroplast (Kazusa)", "Fetch from Kazusa by species ID", "Upload your own codon table"]
kazusa_species_id = "" #@param {type:"string"}
genetic_code = 1 #@param {type:"integer"}
architecture = "9S" #@param ["9S", "14S", "19S"]
synthesis_vendor = "Twist · Standard gene guidelines" #@param ["Twist · Express / Low complexity", "Twist · Standard gene guidelines", "Twist · Complex Genes tolerant", "IDT · gBlocks / eBlocks conservative", "Generic · conservative (default)"]
assembly_enzyme = "GRASP default · BsaI + BpiI + BsmBI" #@param ["GRASP default · BsaI + BpiI + BsmBI", "BsaI (GGTCTC)", "BpiI / BbsI (GAAGAC)", "BsmBI / Esp3I (CGTCTC)", "None (no enzyme filter)"]
site_blacklist = "SapI, BsaI, BpiI" #@param {type:"string"}
#@markdown Extra cut sites to deplete (comma-separated). Default: **SapI, BsaI, BpiI**. Examples: EcoRI, BamHI, HindIII, NotI.
ligation_table = "GRASP stage-matched · L−1 & L1: BsaI-HFv2 · L0: BbsI-HF · 37↔16 °C (Pryor 2020)" #@param ["GRASP stage-matched · L−1 & L1: BsaI-HFv2 · L0: BbsI-HF · 37↔16 °C (Pryor 2020)", "Level 0 override only · T4 ligase 18 h · 25 °C (Potapov 2018; cycling proxy)", "Level 0 override only · T4 ligase 1 h · 25 °C (Potapov 2018)", "Level 0 override only · T4 ligase 1 h · 37 °C (Potapov 2018)", "Level 0 override only · T4 ligase 18 h · 37 °C (Potapov 2018)"]
#@markdown **Plasmid / acceptor overhangs** (typed below): Level −1 entry, Level 0 acceptor outer, Level 1 acceptor outer. Defaults = deposited GRASP toolbox. These are **not** the internal A–E junctions.
#@markdown **Ligation scoring:** one menu covers both enzymes — L−1 & L1 use BsaI-HFv2; Level 0 uses BbsI-HF (BpiI isoschizomer), unless you pick a Potapov Level 0 override.
level_minus1_5prime_overhang = "ACAT" #@param {type:"string"}
level_minus1_3prime_overhang = "ACAA" #@param {type:"string"}
level0_5prime_overhang = "CTCA" #@param {type:"string"}
level0_3prime_overhang = "CTCG" #@param {type:"string"}
level1_5prime_overhang = "GGAG" #@param {type:"string"}
level1_3prime_overhang = "AGCG" #@param {type:"string"}
#@markdown **Plasmid overhangs** = Level −1 / 0 / 1 vector fields below (fixed unless you edit them).
#@markdown **Level 0 overhang redesign** = A 3′ (ACTC), B both, C both, D both, E 5′ (TGAA) only.
redesign_plasmid_overhangs = False #@param {type:"boolean"}
redesign_level0_junctions = False #@param {type:"boolean"}
redesign_selection = "knee" #@param ["knee", "max_fidelity"]
optimize_depth = 2000 #@param {type:"integer"}

from grasp_library import build_default_config, materialize_project
from grasp_library.colab_forms import apply_form_settings
from grasp_library import notebook_ui as ui

PROJECT_DIR = materialize_project()
INPUT_DIR = PROJECT_DIR / "input"
OUTPUT_DIR = PROJECT_DIR / "output"
PROFILE_GB = PROJECT_DIR / "profiles" / "grasp_nar2025" / "genbank"

CONFIG = build_default_config(INPUT_DIR)
CONFIG["project_name"] = "GRASP_library_colab"

if hasattr(ui, "kazusa_codon_reminder"):
    ui.kazusa_codon_reminder()
else:
    ui.note(
        "Package is outdated (missing Kazusa helper). "
        "Re-run <b>0 · Install</b>, then Runtime → Restart session, then Settings again."
    )

applied = apply_form_settings(
    CONFIG,
    organism=organism,
    genetic_code=int(genetic_code),
    target_rna=target_rna,
    architecture=architecture,
    synthesis_vendor=synthesis_vendor,
    assembly_enzyme=assembly_enzyme,
    site_blacklist=site_blacklist,
    ligation_table=ligation_table,
    redesign_plasmid_overhangs=bool(redesign_plasmid_overhangs),
    redesign_level0_junctions=bool(redesign_level0_junctions),
    redesign_selection=redesign_selection,
    level_minus1_5prime_overhang=level_minus1_5prime_overhang,
    level_minus1_3prime_overhang=level_minus1_3prime_overhang,
    level0_5prime_overhang=level0_5prime_overhang,
    level0_3prime_overhang=level0_3prime_overhang,
    level1_5prime_overhang=level1_5prime_overhang,
    level1_3prime_overhang=level1_3prime_overhang,
    optimize_depth=int(optimize_depth),
    kazusa_species_id=kazusa_species_id,
)
CONFIG = applied["config"]
CODON_DATA = applied["codon_data"]
CODON_TABLE = applied["codon_table"]
SELECTED_ORGANISM = organism
ORGANISM_LABEL = applied.get("meta", {}).get("organism", organism)

parts = None
target_map = None
PARETO_FRONT = None
SELECTED_OVERHANGS = None
parts_with_redesigned_junctions = None
optimized_library = None
PARETO_PLOT = None
ASSEMBLY = None
FIDELITY = None

ui.status(
    f"Target <b>{CONFIG['target_rna']}</b> · <b>{ORGANISM_LABEL}</b> · "
    f"genetic code <b>{CONFIG['genetic_code']}</b> · architecture <b>{architecture}</b> · "
    f"plasmid OH redesign <b>{CONFIG['overhang_redesign'].get('plasmid_overhangs')}</b> · "
    f"Level 0 junctions <b>{CONFIG['overhang_redesign'].get('level0_junctions', CONFIG['overhang_redesign'].get('enabled'))}</b> ({redesign_selection}) · "
    f"depth <b>{CONFIG['optimizer']['iterations_per_part']:,}</b><br/>"
    f"Level −1 5′/3′ <code>{level_minus1_5prime_overhang}/{level_minus1_3prime_overhang}</code> · "
    f"Level 0 5′/3′ <code>{level0_5prime_overhang}/{level0_3prime_overhang}</code> · "
    f"Level 1 5′/3′ <code>{level1_5prime_overhang}/{level1_3prime_overhang}</code><br/>"
    f"Ligation model: <b>{CONFIG['ligation']['table_name']}</b><br/>"
    f"Translation QC uses this organism codon table · Project → <code>{PROJECT_DIR}</code>"
)


In [ ]:
#@title 2 · Import GRASP modules { display-mode: "form" }
#@markdown Copies bundled Farley et al. GenBank into the project and builds parts tables.

FORCE_REIMPORT = False #@param {type:"boolean"}

from IPython.display import display
from grasp_library import ensure_grasp_imported
from grasp_library import notebook_ui as ui

if not CODON_DATA:
    ui.note("Run Settings first.")
else:
    imported = ensure_grasp_imported(
        profile_genbank_dir=PROFILE_GB,
        input_dir=INPUT_DIR,
        force=bool(FORCE_REIMPORT),
        log=print,
    )
    parts = imported["parts"]
    target_map = imported.get("target_map")
    display(parts.head())
    jm = imported["junction_map"]
    display(
        jm.groupby("junction", as_index=False)
        .first()[["junction", "native_overhang"]]
    )
    ui.status(
        f"<b>{len(parts)}</b> modules · "
        f"<b>{jm['junction'].nunique()}</b> junctions · "
        f"<b>{len(imported['overhang_candidates'])}</b> deposited fixed-cut candidates; "
        f"runtime ARELF candidates are generated during redesign"
    )


In [ ]:
#@title 3 · Redesign Level 0 junctions { display-mode: "form" }
#@markdown Runs **Level 0 junction redesign only**: A 3′ (ACTC), B both, C both, D both, E 5′ (TGAA). Does **not** change plasmid/acceptor fields (Level −1 / 0 / 1). Checking this enables Level 0 redesign even if Settings left it off.

RUN_LEVEL0_JUNCTION_REDESIGN = True #@param {type:"boolean"}
SEED = 42 #@param {type:"integer"}

import random
import numpy as np
import pandas as pd
from IPython.display import display
from grasp_library import (
    load_and_validate_parts,
    optimize_coding_sequence,
    run_overhang_redesign,
)
from grasp_library import notebook_ui as ui

random.seed(int(SEED))
np.random.seed(int(SEED))

PARETO_FRONT = None
SELECTED_OVERHANGS = None
parts_with_redesigned_junctions = None

if not RUN_LEVEL0_JUNCTION_REDESIGN:
    ui.note("RUN_LEVEL0_JUNCTION_REDESIGN is off — keeping native overhangs.")
elif not CODON_DATA:
    ui.note("Run Settings first.")
else:
    if parts is None and CONFIG["parts_file"].exists():
        parts = load_and_validate_parts(CONFIG["parts_file"])
    # This cell checkbox is authoritative: checking it turns redesign on for
    # the run even if Settings still shows Off.
    redesign_cfg = dict(CONFIG.get("overhang_redesign", {}))
    redesign_cfg["level0_junctions"] = True
    redesign_cfg["enabled"] = True
    CONFIG["overhang_redesign"] = redesign_cfg
    from grasp_library import fidelity_calculator_for_level
    FIDELITY = fidelity_calculator_for_level(
        CONFIG,
        redesign_cfg.get("redesign_level", "level0"),
        min_efficiency=CONFIG.get("ligation", {}).get("min_efficiency", 0.25),
        min_fidelity=CONFIG.get("ligation", {}).get("min_fidelity", 0.9),
    )
    PARETO_FRONT, SELECTED_OVERHANGS, parts_with_redesigned_junctions = run_overhang_redesign(
        parts=parts,
        codon_data=CODON_DATA,
        config=CONFIG,
        optimize_coding_sequence=optimize_coding_sequence,
        input_dir=INPUT_DIR,
        output_dir=OUTPUT_DIR,
        seed=int(SEED),
        fidelity=FIDELITY,
        log=print,
    )
    display(PARETO_FRONT)
    display(
        pd.DataFrame([SELECTED_OVERHANGS], index=["overhang"])
        .T.rename(columns={"overhang": "selected"})
    )
    ui.status("Overhang redesign done — next: anneal library.")


In [ ]:
#@title 4 · Anneal library { display-mode: "form" }
#@markdown Full CDS simulated annealing for every module.

RUN_LIBRARY_OPTIMIZE = True #@param {type:"boolean"}

from IPython.display import display
from grasp_library import (
    load_and_validate_parts,
    optimize_library,
    run_library_optimize,
)
from grasp_library import notebook_ui as ui

optimized_library = None

if not RUN_LIBRARY_OPTIMIZE:
    ui.note("RUN_LIBRARY_OPTIMIZE is off.")
elif not CODON_DATA:
    ui.note("Run Settings first.")
else:
    source_parts = (
        parts_with_redesigned_junctions
        if parts_with_redesigned_junctions is not None
        else parts
    )
    if source_parts is None and CONFIG["parts_file"].exists():
        source_parts = load_and_validate_parts(CONFIG["parts_file"])
    optimized_library = run_library_optimize(
        parts=source_parts,
        codon_data=CODON_DATA,
        config=CONFIG,
        optimize_library=optimize_library,
        output_dir=OUTPUT_DIR,
        log=print,
    )
    if SELECTED_OVERHANGS:
        tag = ";".join(f"{k}={v}" for k, v in sorted(SELECTED_OVERHANGS.items()))
        optimized_library = optimized_library.copy()
        optimized_library["selected_overhangs"] = tag
        optimized_library.to_csv(OUTPUT_DIR / "optimized_library.csv", index=False)
    display(optimized_library.head())
    ui.status(
        f"Annealed <b>{len(optimized_library)}</b> sequences → "
        f"<code>{OUTPUT_DIR / 'optimized_library.csv'}</code><br/>"
        f"Oligo FASTA → <code>{OUTPUT_DIR / 'optimized_grasp_oligos.fasta'}</code>"
    )


In [ ]:
#@title 5 · Pareto plot { display-mode: "form" }
#@markdown Rescores the front after redesign + anneal.

RUN_PARETO_PLOT = True #@param {type:"boolean"}
DEEP_RESCORE_ALL = False #@param {type:"boolean"}

import pandas as pd
from IPython.display import display
from grasp_library import load_and_validate_parts, plot_library_pareto_after_anneal
from grasp_library.workflows import parse_overhang_selection
from grasp_library import notebook_ui as ui

PARETO_PLOT = None

if not RUN_PARETO_PLOT:
    ui.note("RUN_PARETO_PLOT is off.")
elif optimized_library is None or getattr(optimized_library, "empty", False):
    ui.note("Run Anneal library first.")
else:
    front = PARETO_FRONT
    selected = SELECTED_OVERHANGS
    src_parts = parts

    if front is None or getattr(front, "empty", True):
        path = OUTPUT_DIR / "pareto_front.csv"
        if path.exists():
            front = pd.read_csv(path)
            PARETO_FRONT = front
        else:
            ui.note("No Pareto front — run Redesign overhangs first (or enable redesign).")
            front = None

    if front is not None and not front.empty:
        if selected is None and (OUTPUT_DIR / "selected_overhangs.csv").exists():
            sel = pd.read_csv(OUTPUT_DIR / "selected_overhangs.csv")
            if "overhangs" in sel.columns:
                selected = parse_overhang_selection(sel.iloc[0]["overhangs"])
                SELECTED_OVERHANGS = selected
        if src_parts is None and CONFIG["parts_file"].exists():
            src_parts = load_and_validate_parts(CONFIG["parts_file"])
            parts = src_parts

        junction_map = pd.read_csv(INPUT_DIR / "junction_map.csv")
        PARETO_PLOT = plot_library_pareto_after_anneal(
            front=front,
            parts=src_parts,
            junction_map=junction_map,
            codon_data=CODON_DATA,
            config=CONFIG,
            optimized_library=optimized_library,
            selected_overhangs=selected,
            fidelity=FIDELITY,
            deep_all=bool(DEEP_RESCORE_ALL),
            output_dir=OUTPUT_DIR,
            log=print,
        )
        display(PARETO_PLOT["front"])
        display(PARETO_PLOT["figure"])
        chosen = PARETO_PLOT["chosen"]
        SELECTED_OVERHANGS = PARETO_PLOT["selected_overhangs"]
        changed = PARETO_PLOT.get("selection_changed", False)
        ui.status(
            f"Post-anneal best ({CONFIG.get('overhang_redesign', {}).get('selection', 'knee')}) · "
            f"synthesis <b>{float(chosen['synthesis']):.3f}</b> · "
            f"codon <b>{float(chosen['codon_optimality']):.3f}</b> · "
            f"fidelity <b>{float(chosen['ligation_fidelity']):.6f}</b><br>"
            f"{'⚠ Differs from annealed set — re-run Anneal for matching oligos.<br>' if changed else ''}"
            f"Plot → <code>{OUTPUT_DIR / 'pareto_front.png'}</code>"
        )
        display(
            pd.DataFrame([SELECTED_OVERHANGS], index=["overhang"])
            .T.rename(columns={"overhang": "selected_after_anneal"})
        )


In [ ]:
#@title 6 · Export library { display-mode: "form" }
#@markdown Writes CSV / FASTA / Excel and downloads the Excel in Colab.

from grasp_library import export_optimized_library
from grasp_library import notebook_ui as ui

if optimized_library is None:
    ui.note("Run Anneal library first.")
else:
    paths = export_optimized_library(
        optimized_library,
        OUTPUT_DIR,
        selected_overhangs=SELECTED_OVERHANGS,
    )
    ui.status(
        "Exported:<br/>"
        f"• <code>{paths['csv']}</code><br/>"
        f"• <code>{paths['fasta']}</code><br/>"
        f"• <code>{paths['xlsx']}</code>"
    )
    if "google.colab" in __import__("sys").modules:
        from google.colab import files
        files.download(str(paths["xlsx"]))
    for p in sorted(OUTPUT_DIR.glob("optimized_grasp_*")):
        if p.is_file():
            print(p.name)


In [ ]:
#@title 7 · Compile target RNA { display-mode: "form" }
#@markdown GAP-compile the Settings target against the annealed library and stitch the CDS.

RUN_COMPILE = True #@param {type:"boolean"}

from IPython.display import display
from grasp_library import compile_and_assemble_target
from grasp_library import notebook_ui as ui

ASSEMBLY = None

if not RUN_COMPILE:
    ui.note("RUN_COMPILE is off.")
elif optimized_library is None:
    ui.note("Run Anneal library first.")
else:
    import inspect
    _compile_kwargs = dict(
        target_rna=CONFIG["target_rna"],
        optimized_library=optimized_library,
        config=CONFIG,
        input_dir=INPUT_DIR,
        output_dir=OUTPUT_DIR,
        architecture=CONFIG.get("architecture", "9S"),
        five_prime_fusion_site=CONFIG.get("ppr_5prime_fusion_site", "AGGT"),
    )
    if "codon_data" in inspect.signature(compile_and_assemble_target).parameters:
        _compile_kwargs["codon_data"] = CODON_DATA
    else:
        ui.note(
            "Installed package is older than 0.1.1 — re-run <b>0 · Install</b> "
            "for organism codon-table QC on the assembled CDS."
        )
    ASSEMBLY = compile_and_assemble_target(**_compile_kwargs)
    display(ASSEMBLY["assembly_plan"])
    asm = ASSEMBLY["assembled"]
    warn = asm.get("stitch_warning") or ""
    ui.status(
        f"Target <b>{ASSEMBLY['target_rna']}</b> · "
        f"translation verified <b>{asm.get('translation_verified')}</b>"
        + (f"<br/>{warn}" if warn else "")
        + f"<br/>Plan → <code>{ASSEMBLY['plan_csv']}</code>"
        + f"<br/>Assembled CDS FASTA → <code>{ASSEMBLY['assembled_fasta']}</code>"
        + f"<br/>Oligo FASTA → <code>{ASSEMBLY.get('oligo_fasta', '')}</code>"
    )
    if ASSEMBLY.get("oligo_fasta"):
        print("Oligo FASTA:", ASSEMBLY["oligo_fasta"])
        if "google.colab" in __import__("sys").modules:
            from google.colab import files
            files.download(str(ASSEMBLY["oligo_fasta"]))
    for key, value in asm.items():
        if key not in {"assembled_cds", "expected_protein", "observed_protein"}:
            print(f"{key}: {value}")


## Notes

| Step | What happens |
|---|---|
| Install | `pip install grasp-library-designer` from PyPI |
| Import | Bundled GenBank → parts / junctions / candidates |
| Redesign | Synonymous four-base overhangs and movable cuts restricted to ARELF |
| Anneal | Masked SA for every module |
| Compile | GAP part order + CDS stitch |

For a **single binder without the combinatorial library**, open `grasp_oneshot_designer.ipynb`.

Package: https://pypi.org/project/grasp-library-designer/
